# 04. Handoffs (핸드오프 심화)

**핸드오프(Handoff)** 는 에이전트가 특정 작업을 다른 에이전트에게 위임하는 기능입니다.  
핸드오프는 LLM에게 **도구(tool)** 로 표현됩니다.  
예) `Refund Agent`로의 핸드오프 → LLM 도구 이름: `transfer_to_refund_agent`

| 방식 | 예시 | 특징 |
|------|------|------|
| Agent 직접 전달 | `handoffs=[billing_agent]` | 간단, 기본 동작 |
| `handoff()` 함수 | `handoffs=[handoff(billing_agent, on_handoff=...)]` | 콜백, 이름 재정의, 입력 필터 등 고급 옵션 |

**`handoff()` 함수의 주요 파라미터:**

| 파라미터 | 설명 |
|---------|------|
| `agent` | 위임할 대상 에이전트 |
| `tool_name_override` | LLM에게 노출되는 도구 이름 재정의 (기본: `transfer_to_<agent_name>`) |
| `tool_description_override` | 도구 설명 재정의 |
| `on_handoff` | 핸드오프 호출 시 실행되는 콜백 함수 |
| `input_type` | LLM이 핸드오프 시 함께 전달할 데이터 타입 (Pydantic 모델) |
| `input_filter` | 다음 에이전트에게 전달되는 대화 기록 필터링 함수 |
| `is_enabled` | 핸드오프 활성화 여부 (bool 또는 런타임 함수) |

## 1. handoff() 함수 기본 사용법

Agent 인스턴스를 직접 전달하는 방식과 `handoff()` 함수를 사용하는 방식은 **동일하게 동작**합니다.  
`handoff()` 함수 형태는 추가 옵션이 필요할 때 사용합니다.

In [ ]:
# --- 1. 하위 에이전트(Sub-Agent) 정의 ---
# 청구(Billing) 전담 에이전트
# 환불(Refund) 전담 에이전트
# --- 2. 트리아지(분류) 에이전트 정의 ---
# 사용자 요청을 분석한 뒤, 적절한 하위 에이전트로 핸드오프(handoff)하는 역할
        # 방법 1 : Agent 인스턴스를 직접 전달 → 자동으로 tool_name/description 생성
        # 방법 2 : handoff() 함수로 감싸서 전달 → tool 이름·설명을 직접 커스터마이징
# --- 3. 실행 ---
# triage_agent가 입력을 받아 내부적으로 billing_agent 또는 refund_agent로 핸드오프
# 최종 출력 — 핸드오프된 에이전트(billing_agent)가 생성한 응답

## 2. on_handoff 콜백

`on_handoff` 콜백은 핸드오프가 호출되는 순간 실행됩니다.  
데이터 준비, 로깅, 알림 등 사이드 이펙트 처리에 유용합니다.

- `input_type` 없이 사용 시: `def on_handoff(ctx: RunContextWrapper[None])`  
- `input_type` 함께 사용 시: `async def on_handoff(ctx, input_data: MyModel)` (섹션 3 참고)

In [ ]:
# --- 1. 핸드오프 콜백(callback) 함수 정의 ---
# 핸드오프가 실행되는 시점에 자동 호출되는 함수
def on_billing_handoff(ctx: RunContextWrapper[None]):
def on_refund_handoff(ctx: RunContextWrapper[None]):
# --- 2. 하위 에이전트 정의 ---
# 청구 전담 에이전트
# 환불 전담 에이전트
# --- 3. 트리아지(분류) 에이전트 정의 ---
        # on_handoff 파라미터로 콜백 등록
        # 핸드오프 실행 직전에 해당 콜백이 호출됨
# --- 4. 실행 ---
# 입력: "환불을 받고 싶습니다." → triage_agent가 환불 요청으로 분류
# → on_refund_handoff 콜백 실행 → refund_agent로 핸드오프 → 최종 응답 생성

## 3. Handoff 입력 데이터 (input_type)

`input_type`을 사용하면 LLM이 핸드오프 시 **구조화된 데이터를 함께 전달**하도록 할 수 있습니다.  
예를 들어, 에스컬레이션 이유를 함께 전달받아 로깅하거나 처리에 활용할 수 있습니다.

- `input_type`이 있을 때 `on_handoff`는 `async def`로 정의하고 두 번째 인자로 `input_data`를 받습니다.

In [ ]:
# 현재 시간을 문자열로 생성 (에이전트 프롬프트에 사용)
# 에스컬레이션 시 전달할 데이터 구조 정의
# Pydantic 모델을 사용하면 에이전트가 전달해야 하는 입력 형식을 강제할 수 있음
class EscalationData(BaseModel):
# 에스컬레이션이 발생했을 때 실행되는 콜백 함수
# ctx : 실행 컨텍스트
# input_data : 에이전트가 전달한 EscalationData 객체
# 복잡한 문제를 처리하는 전문 상담 에이전트
# Support agent가 처리하지 못하는 문제를 넘겨받아 해결
# 일반 고객 지원을 처리하는 1차 상담 에이전트
    # 다른 에이전트로 작업을 넘기는 handoff 설정
# Support agent 실행
# 사용자의 입력을 전달하면 필요 시 escalation_agent로 handoff 발생
# 최종 에이전트 응답 출력

## 4. Input Filter (입력 필터)

핸드오프가 발생하면 새로운 에이전트는 기존 대화 기록 전체를 받습니다.  
`input_filter`를 사용하면 다음 에이전트에게 전달되는 **대화 기록을 가공**할 수 있습니다.

`agents.extensions.handoff_filters`에는 자주 사용되는 내장 필터가 제공됩니다:

| 필터 | 설명 |
|------|------|
| `remove_all_tools` | 대화 기록에서 모든 도구 호출/결과 제거 |
|`nest_handoff_history` | 이전 대화 내용을 요약해서 단일 assistant 메시지로 압축 후 전달 |

### remove_all_tools는 언제 쓰나요?

받는 쪽 에이전트에게 **도구 기록이 필요 없거나 오히려 해가 될 때** 사용합니다.

| 상황 | 예시 |
|---|---|
| **민감 정보 차단** | 고객 DB 조회 결과(주민번호, 카드번호, 내부 API 응답)를 외부 상담용 에이전트에 넘기지 않을 때 |
| **토큰 비용 절감** | RAG 검색 문서나 웹 검색 결과처럼 긴 도구 출력이 다음 에이전트에는 필요 없을 때 |
| **혼란 방지** | 받는 에이전트에 없는 도구의 호출 기록이 남아 있으면, 그 도구를 쓰려 하거나 결과를 잘못 해석할 수 있음 |
| **역할 분리** | 앞 에이전트의 작업 과정은 숨기고, 대화 내용만으로 새로 판단하게 할 때 |

### 흐름 설명
```
사용자: "주문번호 ORD-1234 배송 조회해주고, 반품 정책도 알려줘"
      ↓
main_agent
  → check_shipping_status("ORD-1234") 도구 호출  ← 도구 메시지 생성
  → "반품 정책은 FAQ agent한테 넘겨야겠다"
  → remove_all_tools 필터 실행
      ┌─────────────────────────────────────┐
      │ 도구 호출 기록 제거 전                 │
      │ [user] 배송 조회해주고 반품도 알려줘    │
      │ [tool_call] check_shipping_status   │  ← 제거됨
      │ [tool_result] 배송 중...             │  ← 제거됨
      └─────────────────────────────────────┘
      ↓
faq_agent (도구 기록 없이 깔끔한 대화만 전달)
  → 반품 정책 답변
```

### nest_handoff_history는 언제 쓰나요?

핸드오프할 때 **앞 에이전트의 작업 기록을 지우지 않고, 압축해서 넘기고 싶을 때** 사용합니다.

- 도구 호출, 도구 결과 같은 기록을 `<CONVERSATION HISTORY>`로 감싼 **assistant 메시지(요약 구간)** 로 정리해 전달합니다.
- 사용자 메시지 같은 일반 메시지는 원래 위치에 그대로 둡니다.
- **opt-in 베타 기능**이라 기본값은 꺼져 있습니다(`False`). 꺼져 있으면 원본 기록을 그대로 넘깁니다.

| 상황 | 예시 |
|---|---|
| **정보는 유지하면서 기록 정리** | 배송 조회 결과는 FAQ 에이전트가 알아야 하지만, 도구 호출 형식 그대로 넘길 필요는 없을 때 |
| **혼란 방지** | 받는 에이전트에 없는 도구의 호출 항목 대신, 읽기 쉬운 텍스트로 이전 경과를 전달할 때 |
| **여러 단계 핸드오프** | A → B → C로 핸드오프가 이어질 때, 이전 요약 구간을 다시 펼쳐 순서대로 재구성해 넘길 때 |

### 흐름 설명
```
사용자: "주문번호 ORD-1234 배송 조회해주고, 반품 정책도 알려줘"
      ↓
main_agent
  → check_shipping_status("ORD-1234") 도구 호출  ← 도구 메시지 생성
  → "반품 정책은 FAQ agent한테 넘겨야겠다"
  → nest_handoff_history 실행 (RunConfig 또는 handoff에서 True)
      ┌──────────────────────────────────────────┐
      │ 정리 전                                   │
      │ [user] 배송 조회해주고 반품도 알려줘         │  ← 그대로 유지
      │ [tool_call] check_shipping_status        │  ┐
      │ [tool_result] 배송 중 (예상 도착: 내일)     │  ┘ 텍스트로 정리
      └──────────────────────────────────────────┘
      ┌──────────────────────────────────────────┐
      │ 정리 후                                   │
      │ [user] 배송 조회해주고 반품도 알려줘         │
      │ [assistant] <CONVERSATION HISTORY>       │
      │   check_shipping_status 호출 →           │
      │   배송 중 (예상 도착: 내일)                 │
      │ </CONVERSATION HISTORY>                  │
      └──────────────────────────────────────────┘
      ↓
faq_agent (도구 결과를 텍스트로 확인 가능)
  → "ORD-1234는 배송 중, 내일 도착 예정" + 반품 정책 답변
```

> `remove_all_tools`와 달리 도구 결과가 **삭제되지 않고 텍스트로 남기 때문에**, faq_agent가 배송 상태까지 함께 안내할 수 있습니다.
> 단, `input_filter`가 설정된 핸드오프에는 적용되지 않습니다.

In [ ]:
# -------------------------------
# 도구(tool) 정의
# -------------------------------
def check_shipping_status(order_id: str) -> str:
    # 실제 시스템에서는 DB 또는 API 조회
# -------------------------------
# FAQ 에이전트 정의
# -------------------------------
    # FAQ 관련 질문을 담당하는 에이전트
# -------------------------------
# 메인 에이전트 생성 함수
# -------------------------------
# mode: "none"   → 원본 기록 그대로 전달 (기본 동작)
#       "remove" → remove_all_tools로 도구 호출/결과 삭제 후 전달
#       "nest"   → nest_handoff_history로 도구 기록을 텍스트 요약 구간으로 정리해 전달
def build_main_agent(mode: str = "none") -> Agent:
        # 역할 정의
        # 이 에이전트가 사용할 수 있는 Tool 목록
        # 다른 에이전트에게 작업을 넘기는 handoff 설정
                # remove 모드에서만 도구 기록 제거 필터 적용
                # nest 모드에서만 기록 압축 적용 (None이면 RunConfig 설정을 따름, 기본은 꺼짐)
# -------------------------------
# 에이전트 실행 함수
# -------------------------------
        # 배송조회 + FAQ 질문이 동시에 포함된 요청
    # 최종 에이전트 응답 출력
# 세 가지 방식 비교 실행

---------------------
## 5. 권장 프롬프트 (RECOMMENDED_PROMPT_PREFIX)

LLM이 핸드오프를 올바르게 이해하고 활용하려면 관련 정보를 프롬프트에 포함시키는 것이 좋습니다.  
```
LLM은 핸드오프가 뭔지 기본적으로 모릅니다. 그래서 "너는 다른 에이전트에게 작업을 넘길 수 있어" 라고 미리 알려줘야 합니다.

핸드오프 설명 없이:
  LLM: "나는 그냥 질문에 답하는 AI야"
  → 핸드오프 도구가 있어도 잘 안 씀

핸드오프 설명 포함:
  LLM: "나는 다른 전문 에이전트에게 작업을 위임할 수 있어"
  → 핸드오프를 적극적으로 활용
```
`agents.extensions.handoff_prompt`에서 권장 프롬프트를 제공합니다:

| 방법 | 설명 |
|------|------|
| `RECOMMENDED_PROMPT_PREFIX` | 프롬프트 앞에 직접 붙여 사용하는 문자열 상수 |
| `prompt_with_handoff_instructions(prompt)` | 기존 프롬프트에 권장 내용을 자동으로 추가하는 함수 |

```
# 시스템 컨텍스트
당신은 Agents SDK라는 멀티 에이전트 시스템의 일부입니다.
이 시스템은 에이전트 간의 조율과 실행을 쉽게 만들기 위해 설계되었습니다.

Agents SDK는 두 가지 핵심 추상화를 사용합니다: **에이전트(Agents)** 와 **핸드오프(Handoffs)**.

에이전트는 지시사항(instructions)과 도구(tools)를 포함하며,
필요한 경우 대화를 다른 에이전트에게 넘길 수 있습니다.

핸드오프는 일반적으로 `transfer_to_<에이전트명>` 형태로 명명된
핸드오프 함수를 호출함으로써 이루어집니다.

에이전트 간의 전환은 백그라운드에서 원활하게 처리되므로,
사용자와의 대화에서 이러한 전환을 언급하거나 드러내지 마세요.
```

In [ ]:
# 청구/결제 전문 에이전트
# 환불 전문 에이전트
# 사용자 요청을 분석하여 적절한 전문 에이전트로 라우팅하는 triage 에이전트
# triage_agent 실행 → 청구 관련 질문이므로 billing_agent로 핸드오프 예상

## 6. 종합 예제 - 고객 지원 시스템

여러 핸드오프 기능을 조합한 고객 지원 시스템입니다.

```
사용자
  └─→ Triage Agent (분류)
        ├─→ Order Agent    (주문 조회)
        ├─→ Refund Agent   (환불 처리, on_handoff + input_type)
        └─→ FAQ Agent      (일반 문의, input_filter 적용)
```

In [ ]:
# 환불 요청 데이터 구조 정의 (LLM이 핸드오프 시 채워서 전달)
class RefundRequest(BaseModel):
# 환불 핸드오프 발생 시 실행되는 콜백 함수 (로깅 용도)
# 주문/배송 전문 에이전트
# 환불 전문 에이전트
# FAQ 전문 에이전트
# 고객 문의를 분류하여 적절한 전문 에이전트로 라우팅하는 triage 에이전트

### 실습 문제

아래 요구사항에 맞는 **여행 예약 지원 시스템**을 구현하세요.

**에이전트 구성:**

1. **Triage Agent**: 사용자 요청을 분류하여 적절한 에이전트에 핸드오프
2. **Flight Agent**: 항공편 예약 및 조회 처리
3. **Hotel Agent**: 호텔 예약 및 조회 처리
4. **Cancellation Agent**: 예약 취소 처리 (`on_handoff` 콜백으로 취소 정보 로깅)

**요구사항:**
- `Cancellation Agent`로 핸드오프 시 `input_type`으로 예약 번호(`booking_id: str`)와 취소 사유(`reason: str`)를 전달받아 출력
- 모든 에이전트에 `prompt_with_handoff_instructions` 적용
- `Flight Agent`, `Hotel Agent`는 `handoff()` 함수 형태로 지정

**테스트 입력:**
- `"서울-제주 항공편을 예약하고 싶습니다."` → Flight Agent 응답
- `"제주도 호텔을 3박 예약하려고 합니다."` → Hotel Agent 응답
- `"예약번호 BK-999 항공편을 취소하고 싶어요. 일정이 바뀌어서요."` → Cancellation Agent 핸드오프 + 콜백 출력